In [1]:
import pandas as pd 
import json
import bw2data as bd
import bw2calc as bc

# import own Python files, vars, mappings, and functions
from config import (CC_METHOD, NAME_REF_DB, NAME_FUTURE_DB,
                    COST_DATA, PROJECT_NAME, OUT_JSON_GHG, OUT_JSON_GHG_FUTURE)
import create_db_lca_functions as lcaf

13:11:11+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
GENERATE_NEW_LCA_DB=False
CALC_ALL_LCA_IMPACTS=True

In [3]:
PROJECT_NAME

'ecoinvent-3.12-cutoff_v2'

import brightway2 as bw
if 'biosphere3' in list(bw.databases):
    print("Deleting existing database...")
    del bw.databases['biosphere3']
bw.databases

In [5]:
bd.projects.set_current(PROJECT_NAME)

# If we want to use a full LCA approach, we have to set-up the LCA database:
if GENERATE_NEW_LCA_DB:
    bd.projects.set_current(PROJECT_NAME)
    # Import LCIA methods, import ecoinvent cut-off and consequential dbs
    #lcaf.import_additional_lcias()
    lcaf.import_ecoinvent_database()
    # Generate reference database with premise to import additional novel LCIs.
    lcaf.generate_reference_database()

    # Make future scenario, use 2C scenario from REMIND
    list_spec_scenarios, list_names = lcaf.generate_future_ei_dbs(scenarios = ["SSP2-PkBudg1000"], iam = 'remind',
                                       start_yr=2025, end_yr = 2050, step = 25, endstring="base")
    lcaf.generate_prospective_lca_dbs(list_spec_scenarios, list_names)

# Generate GHG emission data
if CALC_ALL_LCA_IMPACTS:
    cost_dict = COST_DATA[NAME_REF_DB].to_dict() # techno-economic data
    dict_ghg_impacts = lcaf.get_tech_environmental_burdens(cost_dict) 
    with open("input_data/dict_ghg_impacts.txt", 'w') as file:
        json.dump(dict_ghg_impacts, file)
        
    cost_dict_future = COST_DATA[NAME_FUTURE_DB].to_dict() # techno-economic data
    dict_ghg_impacts_future = lcaf.get_tech_environmental_burdens(cost_dict_future, sec_db=NAME_FUTURE_DB) 
    with open("input_data/dict_ghg_impacts_future.txt", 'w') as file:
        json.dump(dict_ghg_impacts_future, file)
else:
    #Otherwise, just used the stored data valid for ecoinvent cut-off
    with open("input_data/dict_ghg_impacts.txt", 'r') as file:
        dict_ghg_impacts = json.load(file)
        
    with open("input_data/dict_ghg_impacts_future.txt", 'r') as file:
        dict_ghg_impacts_future = json.load(file)
dict_ghg_impacts

{'ghg_imp_h2_ves': 0.0,
 'ghg_imp_pv': 0.0,
 'ghg_imp_wind_on': 0.0,
 'ghg_imp_electr': 0.0,
 'ghg_imp_bat_cap': 0.0,
 'ghg_impact_grid_network': 0.0,
 'ghg_imp_asu': 0.0,
 'ghg_imp_hb': 0.008437000215053558}

In [4]:
import bw2data as bd
import bw2io as bi
from config import PROJECT_NAME, NAME_REF_DB, NAME_FUTURE_DB, BIOSPHERE_DB

assert PROJECT_NAME in bd.projects
bd.projects.set_current(PROJECT_NAME)

for target in dict.fromkeys([NAME_REF_DB, NAME_FUTURE_DB]):
    assert target in bd.databases, f"Missing database: {target}"

    imp = bi.ExcelImporter(r"input_data\lci-add.xlsx")

    # Assign the imported activities to this target database.
    imp.db_name = target
    for dataset in imp.data:
        dataset["database"] = target

    imp.apply_strategies()
    imp.match_database(
        fields=["name", "reference product", "unit", "location"]
    )
    imp.match_database(
        target,
        fields=["name", "reference product", "unit", "location"],
        edge_kinds=["technosphere"],
    )
    imp.match_database(
        BIOSPHERE_DB,
        fields=["name", "unit", "categories"],
        edge_kinds=["biosphere"],
    )

    imp.statistics()
    unlinked = list(imp.unlinked)
    if unlinked:
        print(unlinked[:10])
        raise RuntimeError(f"Resolve unlinked exchanges in {target} first")

    existing_codes = {act["code"] for act in bd.Database(target)}
    if any(ds["code"] in existing_codes for ds in imp.data):
        raise RuntimeError(
            f"Activity code collision in {target}; inspect first"
        )

    imp.write_database(delete_existing=False)
    print(f"Added inventories to {target}")

Extracted 1 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 14 strategies in 0.06 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Graph statistics for `ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030_08_full` importer:
2 graph nodes:
	None: 2
18 graph edges:
	biospher

100%|██████████| 42510/42510 [00:20<00:00, 2122.23it/s]


13:17:53+0200 [info     ] Vacuuming database            
Created database: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030_08_full
Added inventories to ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030_08_full
Extracted 1 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 14 strategies in 0.07 seconds
Applying strategy: link_iterable_by_fields
Applying strate

100%|██████████| 42512/42512 [00:14<00:00, 3006.84it/s]


13:34:45+0200 [info     ] Vacuuming database            
Created database: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2050_08_full
Added inventories to ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2050_08_full


In [4]:
print("Project:", lcaf.bd.projects.current)
print("Requested method:", CC_METHOD)
print("Available:", CC_METHOD in lcaf.bd.methods)

for method in lcaf.bd.methods:
    if "IPCC" in str(method).upper():
        print(method)

Project: ecoinvent-3.12-cutoff_v2
Requested method: ('ecoinvent-3.12', 'IPCC 2021 (incl. biogenic CO2)', 'climate change: total (incl. biogenic CO2, incl. SLCFs)', 'global warming potential (GWP100)')
Available: False
('IPCC 2013 no LT', 'climate change no LT', 'global temperature change potential (GTP100) no LT')
('IPCC 2013 no LT', 'climate change no LT', 'global temperature change potential (GTP20) no LT')
('IPCC 2013 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')
('IPCC 2013 no LT', 'climate change no LT', 'global warming potential (GWP20) no LT')
('IPCC 2013', 'climate change', 'global temperature change potential (GTP100)')
('IPCC 2013', 'climate change', 'global temperature change potential (GTP20)')
('IPCC 2013', 'climate change', 'global warming potential (GWP100)')
('IPCC 2013', 'climate change', 'global warming potential (GWP20)')
('IPCC 2021 (incl. biogenic CO2) no LT', 'climate change: biogenic (incl. CO2) no LT', 'global warming potential (GWP1

In [7]:
print("Requested:", NAME_REF_DB)
print("Available cost columns:", COST_DATA.columns.tolist())

Requested: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030_08_full
Available cost columns: ['ecoinvent_312_reference', 'ecoinvent_remind_SSP2-PkBudg1000_2050_base']


In [5]:
print("Project:", lcaf.bd.projects.current)
print("Database:", NAME_REF_DB)
print("Activity count:", len(lcaf.bd.Database(NAME_REF_DB)))

for act in lcaf.bd.Database(NAME_REF_DB):
    name = act.get("name", "").lower()
    if "hydrogen" in name and ("storage" in name or "tank" in name):
        print(
            act["name"],
            act.get("location"),
            act.get("reference product"),
        )

Project: ecoinvent-3.12-cutoff_v2
Database: ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2030_08_full
Activity count: 42508
electricity production, at hydrogen-fired combined cycle power plant, by auto-thermal reforming of natural gas, pre, pipeline 200km, storage 1000m RER electricity, high voltage
carbon dioxide storage at hydrogen production plant, pre, pipeline 200km, storage 1000m IND carbon dioxide storage at hydrogen production plant, pre, pipeline 200km, storage 1000m
carbon dioxide storage at hydrogen production plant, pre, pipeline 200km, storage 1000m EUR carbon dioxide storage at hydrogen production plant, pre, pipeline 200km, storage 1000m
carbon dioxide storage at hydrogen production plant, pre, pipeline 400km, storage 3000m JPN carbon dioxide storage at hydrogen production plant, pre, pipeline 400km, storage 3000m
carbon dioxide, captured at hydrogen production plant, pre, pipeline 200km, storage 1000m RER carbon dioxide, captured at hydrogen production plant, pre, pipeli

In [5]:
dict_ghg_impacts_future

{'ghg_imp_h2_ves': 6.5074787351482,
 'ghg_imp_pv': 316.98510826548363,
 'ghg_imp_wind_on': 301.2385348433775,
 'ghg_imp_electr': 29.365497666746304,
 'ghg_imp_bat_cap': 30.003363612916782,
 'ghg_impact_grid_network': 26.17529236016685,
 'ghg_imp_asu': 6.962708526492677e-05,
 'ghg_imp_hb': 0.020867579095956316}

## Generate pre-calculated GHG emission factors for the grid

In [6]:
metadata_list = []
for db in [NAME_REF_DB, NAME_FUTURE_DB]:
    all_acts = [act for act in bd.Database(db) if ('market for electricity, low voltage' == act['name'] or 'market group for electricity, low voltage' == act['name']) and 'electricity, low voltage' == act['reference product'] ] 
    
    for act_sel in all_acts:
        """Store metadata"""
        metadata = {
            'location': act_sel.get('location', ''),
            "db": db,
            'name': act_sel.get('name', ''),
            'unit': act_sel.get('unit', ''),
            'reference_product': act_sel.get('reference product', ''),
            'key': act_sel.key,
            
        }
        metadata_list.append(metadata)

df_meta = pd.DataFrame(metadata_list)
df_meta

,location,db,name,unit,reference_product,key
0,US-MRO,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
1,CN-NCGC,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
2,UA,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
3,Europe without Switzerland,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market group for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
4,CN-NECG,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
...,...,...,...,...,...,...
405,BH,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
406,FI,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
407,CD,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...
408,ZW,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...


In [7]:

def run_mlca(
    activity_keys: list,
    functional_units: list,
    result_index_labels: list,
    column_suffix: str,
    impact_methods: list,
) -> pd.DataFrame:
    """
    Run a BW2.5 MultiLCA calculation and return a DataFrame of results.

    Parameters
    ----------
    activity_keys : list
        Brightway activity references. Can be:
        - integer node ids
        - (database, code) tuples
        - Activity objects
    functional_units : list
        Functional unit amounts.
    result_index_labels : list
        Labels to keep in the output table, e.g. Brightway keys.
    column_suffix : str
        Suffix for result columns.
    impact_methods : list
        List of LCIA method tuples.

    Returns
    -------
    pd.DataFrame
        DataFrame with a normal 'key' column, not an index.
    """
    if not (len(activity_keys) == len(functional_units) == len(result_index_labels)):
        raise ValueError(
            "activity_keys, functional_units, and result_index_labels must have the same length."
        )

    def to_node_id(obj):
        if isinstance(obj, int):
            return obj
        elif isinstance(obj, tuple) and len(obj) == 2:
            return bd.get_node(database=obj[0], code=obj[1]).id
        elif hasattr(obj, "id"):
            return obj.id
        else:
            raise TypeError(
                f"Unsupported activity key type: {type(obj)}. "
                "Use int ids, (database, code) tuples, or Activity objects."
            )

    # safe internal labels for MultiLCA
    demand_labels = [f"fu_{i}" for i in range(len(activity_keys))]

    demands = {
        demand_label: {to_node_id(key): float(fu)}
        for demand_label, key, fu in zip(demand_labels, activity_keys, functional_units)
    }

    method_config = {"impact_categories": impact_methods}

    data_objs = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=method_config,
    )

    mlca = bc.MultiLCA(
        demands=demands,
        method_config=method_config,
        data_objs=data_objs,
    )
    mlca.lci()
    mlca.lcia()

    suffix = column_suffix if (column_suffix == "" or column_suffix.startswith("_")) else f"_{column_suffix}"
    colnames = [f"lca_impact{suffix}_{method[-1]}" for method in impact_methods]

    rows = []
    for i, demand_label in enumerate(demand_labels):
        row = {"key": result_index_labels[i]}
        for method in impact_methods:
            col = f"lca_impact{suffix}_{method[-1]}"
            row[col] = mlca.scores[(method, demand_label)]
        rows.append(row)

    return pd.DataFrame(rows)

In [8]:
CC_METHOD

('IPCC 2021', 'climate change', 'GWP 100a, incl. H')

In [9]:
def run_mlca(
    activity_keys: list,
    functional_units: list,
    result_index_labels: list,
    column_suffix: str,
    impact_methods: list,
) -> pd.DataFrame:
    """
    Run a BW2.5 MultiLCA calculation and return a DataFrame of results.

    Parameters
    ----------
    activity_keys : list
        Brightway activity references. Can be:
        - integer node ids
        - (database, code) tuples
        - Activity objects
    functional_units : list
        Functional unit amounts.
    result_index_labels : list
        Labels to keep in the output table, e.g. Brightway keys.
    column_suffix : str
        Suffix for result columns.
    impact_methods : list
        List of LCIA method tuples.

    Returns
    -------
    pd.DataFrame
        DataFrame with a normal 'key' column, not an index.
    """
    if not (len(activity_keys) == len(functional_units) == len(result_index_labels)):
        raise ValueError(
            "activity_keys, functional_units, and result_index_labels must have the same length."
        )

    def to_node_id(obj):
        if isinstance(obj, int):
            return obj
        elif isinstance(obj, tuple) and len(obj) == 2:
            return bd.get_node(database=obj[0], code=obj[1]).id
        elif hasattr(obj, "id"):
            return obj.id
        else:
            raise TypeError(
                f"Unsupported activity key type: {type(obj)}. "
                "Use int ids, (database, code) tuples, or Activity objects."
            )

    # safe internal labels for MultiLCA
    demand_labels = [f"fu_{i}" for i in range(len(activity_keys))]

    demands = {
        demand_label: {to_node_id(key): float(fu)}
        for demand_label, key, fu in zip(demand_labels, activity_keys, functional_units)
    }

    method_config = {"impact_categories": impact_methods}

    data_objs = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=method_config,
    )

    mlca = bc.MultiLCA(
        demands=demands,
        method_config=method_config,
        data_objs=data_objs,
    )
    mlca.lci()
    mlca.lcia()

    suffix = column_suffix if (column_suffix == "" or column_suffix.startswith("_")) else f"_{column_suffix}"
    colnames = [f"lca_impact{suffix}_{method[-1]}" for method in impact_methods]

    rows = []
    for i, demand_label in enumerate(demand_labels):
        row = {"key": result_index_labels[i]}
        for method in impact_methods:
            col = f"lca_impact{suffix}_{method[-1]}"
            row[col] = mlca.scores[(method, demand_label)]
        rows.append(row)

    return pd.DataFrame(rows)

df_lca = run_mlca(
    activity_keys=df_meta["key"].tolist(),
    functional_units=[1] * len(df_meta),
    result_index_labels=df_meta["key"].tolist(),
    column_suffix="",
    impact_methods=[CC_METHOD]
)

df_total_power = df_meta.merge(df_lca, on="key", how="left")
df_total_power

,location,db,name,unit,reference_product,key,"lca_impact_GWP 100a, incl. H"
0,US-MRO,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
1,CN-NCGC,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
2,UA,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
3,Europe without Switzerland,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market group for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
4,CN-NECG,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
...,...,...,...,...,...,...,...
405,BH,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
406,FI,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
407,CD,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0
408,ZW,ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_20...,"market for electricity, low voltage",kilowatt hour,"electricity, low voltage",(ecoinvent-3.12-cutoff_remind-SSP1-PkBudg650_2...,0.0


In [10]:
df_total_power_2025=df_total_power[df_total_power['db'] == NAME_REF_DB]
df_total_power_2050=df_total_power[df_total_power['db'] == NAME_FUTURE_DB]

# Convert to dictionary indexed by 'location'
total_power_dict = df_total_power_2025.set_index('location').to_dict(orient='index')
total_power_dict_future = df_total_power_2050.set_index('location').to_dict(orient='index')

# Save to JSON file
with open(OUT_JSON_GHG, "w") as f:
    json.dump(total_power_dict, f, indent=2)

with open(OUT_JSON_GHG_FUTURE, "w") as f:
    json.dump(total_power_dict_future, f, indent=2)